# Lexicon Personality Bot: notebook agent (v3)
Writes chat replies as one specific person (P1 or P2) using a personal RAG database of that person's own replies.
No fine-tuning. Sections:
1. Config and setup  2. Load and merge survey data  3. Seeded random test split (frozen)  4. Structured Outputs tagging and frozen gists
5. Style card  6. RAG database (MiniLM + StyleDistance)  7. Generation (conditions A, B, C, D, E)  8. Judge calibration
9. Evaluation (pairwise style judge, meaning judge, copy rate, person classifier, bootstrap CIs)  10. The agent (understand, retrieve, draft, check, learn)
11. Progressive augmentation (free ChatGPT, confirm / edit / reject)

Every LLM call is cached in `cache/llm_cache.jsonl`, so re-running the notebook costs nothing and gives the same numbers.

## 1. Config

In [ ]:
# pip install openai sentence-transformers scikit-learn rapidfuzz pandas numpy
import os, json, re, hashlib, random, math
from pathlib import Path
import numpy as np, pandas as pd

USER_IDS = ["P1", "P2"]
RESPONSE_FILES = {"P1": "data/P1_responses.csv", "P2": "data/P2_responses.csv"}
BANK_CSV = "data/prompt_bank_v2.csv"
LABELS_CSV = "data/labels.csv"
SEED = 1
TEST_PER_LABEL = 2
INCLUDE_SURVEY_REGISTER = False   # P1's "reply properly" answers were rushed (4-7 s, 1-2 words); keep them out of the database
MODEL = "gpt-4o-mini"
PRICE = {"in": 0.15/1e6, "out": 0.60/1e6}   # $ per token for MODEL
BUDGET_USD = 1.00                 # hard stop for this notebook
GEN_TEMPERATURE, GEN_MAX_TOKENS = 0.7, 30
K = 3
SEM_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
STYLE_MODEL = "StyleDistance/styledistance"
CODE_DIR = "code" if Path("code").exists() else "../survey_v2/code"   # lexicon_gen_judge.py (MLP Fine-tuner) and lexicon_retrieval.py (Researcher)
for p in ["cache", "frozen", "db", "results", "augment"]: Path(p).mkdir(exist_ok=True)

In [ ]:
import sys; sys.path.insert(0, CODE_DIR)
from types import SimpleNamespace as NS
from openai import OpenAI
_real = OpenAI()   # reads OPENAI_API_KEY from the environment; never print it
CACHE_FILE = Path("cache/llm_cache.jsonl")
_cache = {}
if CACHE_FILE.exists():
    for line in CACHE_FILE.read_text().splitlines():
        r = json.loads(line); _cache[r["k"]] = r["v"]
SPEND = {"usd": 0.0, "calls": 0, "cached": 0}

class _Completions:
    def create(self, **kw):
        """Drop-in for client.chat.completions.create: caches every call and enforces BUDGET_USD."""
        key = hashlib.sha256(json.dumps(kw, sort_keys=True, default=str).encode()).hexdigest()
        if key in _cache:
            SPEND["cached"] += 1; text = _cache[key]
        else:
            if SPEND["usd"] > BUDGET_USD: raise RuntimeError(f"Budget cap reached: ${SPEND['usd']:.3f}")
            r = _real.chat.completions.create(**kw); u = r.usage
            SPEND["usd"] += u.prompt_tokens*PRICE["in"] + u.completion_tokens*PRICE["out"]; SPEND["calls"] += 1
            text = r.choices[0].message.content; _cache[key] = text
            with CACHE_FILE.open("a") as f: f.write(json.dumps({"k": key, "v": text}) + "\n")
        return NS(choices=[NS(message=NS(content=text))], usage=NS(total_tokens=0, prompt_tokens=0, completion_tokens=0))
client = NS(chat=NS(completions=_Completions()))   # pass this to every module function

def llm(messages, schema=None, name="out", temperature=0.0, max_tokens=300):
    kw = dict(model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens, seed=SEED)
    if schema: kw["response_format"] = {"type": "json_schema", "json_schema": {"name": name, "schema": schema, "strict": True}}
    text = client.chat.completions.create(**kw).choices[0].message.content
    return json.loads(text) if schema else text.strip()

def obj(props, req=None):
    return {"type": "object", "properties": props, "required": req or list(props), "additionalProperties": False}

## 2. Load and merge survey data
Labels and scene metadata live only in the bank (the survey app was blind), so they are merged back by `scene_id`. Teammate names in the shown message are replaced by `TM`. Pilot rows are dropped.

In [ ]:
bank = pd.read_csv(BANK_CSV)
LABELS = pd.read_csv(LABELS_CSV)
LABEL_NAMES = LABELS.target_label.tolist()
NEG = set(LABELS[LABELS.is_negative].target_label)

def load_user(uid):
    d = pd.read_csv(RESPONSE_FILES[uid])
    d = d[d["round"].astype(str) != "pilot"].copy()
    d["user_id"] = uid
    d = d.drop(columns=[c for c in ["target_label", "split"] if c in d]).merge(
        bank[["scene_id", "target_label", "sender_role", "stakes", "context"]], on="scene_id", how="left")
    d["incoming"] = [str(m).replace(f"{n}:", "TM:") if isinstance(n, str) else m for m, n in zip(d.shown_message, d.teammate_filled)]
    d["abstain"] = d["abstain"].astype(str).str.lower().eq("true") | d.response.isna()
    d["response"] = d.response.fillna("").astype(str).str.strip()
    d["row_id"] = [f"{uid}_{t}_{s if isinstance(s,str) else f}_{r}" for t, s, f, r in zip(d.item_type, d.scene_id, d.free_id, d.register)]
    return d

data = pd.concat([load_user(u) for u in USER_IDS], ignore_index=True)
chat = data[(data.item_type == "scene") & (data.register == "chat")]
print(data.groupby(["user_id", "item_type", "register"]).size())
print("answered chat scenes per label (min):", chat[~chat.abstain].groupby(["user_id","target_label"]).size().min())

## 3. Seeded random test split (frozen)
Per person: 2 answered chat scenes per label (40). Abstain rows are skipped. A scene's survey-register twin follows it into the test side. Once written, the split file is never redrawn.

In [ ]:
def make_split(uid, seed=SEED):
    f = Path(f"frozen/split_{uid}_seed{seed}.json")
    if f.exists():
        return json.loads(f.read_text())
    c = chat[(chat.user_id == uid) & (~chat.abstain)]
    rng = random.Random(seed)
    test = []
    for lab in LABEL_NAMES:
        ids = sorted(c[c.target_label == lab].scene_id)
        test += rng.sample(ids, min(TEST_PER_LABEL, len(ids)))
    split = {"user_id": uid, "seed": seed, "test_scene_ids": sorted(test)}
    f.write_text(json.dumps(split, indent=1))
    return split

SPLITS = {u: make_split(u) for u in USER_IDS}

def db_rows(uid):
    d = data[(data.user_id == uid) & (~data.abstain) & (data.item_type == "scene")]
    d = d[~d.scene_id.isin(SPLITS[uid]["test_scene_ids"])]          # twin rule: both registers leave together
    if not INCLUDE_SURVEY_REGISTER: d = d[d.register == "chat"]
    return d

def test_rows(uid):
    return chat[(chat.user_id == uid) & chat.scene_id.isin(SPLITS[uid]["test_scene_ids"])]

for u in USER_IDS:
    print(u, "test:", len(test_rows(u)), "database:", len(db_rows(u)))

## 4. Structured Outputs tagging and frozen gists
Every example gets one strict-JSON call. For test items, `intended_meaning` becomes the frozen gist: neutral, third person, meaning only, and it must share no 4-word sequence with the real reply. **Review `frozen/gists_*.csv` by hand once, then leave it.**

In [ ]:
TAG_SCHEMA = obj({
    "scenario": {"type": "string", "enum": ["plans", "class_project", "logistics", "personal_news", "opinion", "work"]},
    "sender_role": {"type": "string", "enum": ["friend", "teammate", "authority"]},
    "stakes": {"type": "string", "enum": ["trivial", "social", "class", "work"]},
    "intent": {"type": "string", "enum": LABEL_NAMES},
    "intended_meaning": {"type": "string"},
    "tone": {"type": "string", "enum": ["warm", "neutral", "blunt", "annoyed", "playful", "sarcastic"]},
    "register": {"type": "string", "enum": ["chat", "email", "formal"]},
    "length_bucket": {"type": "string", "enum": ["short", "medium", "long"]},
})
LABEL_DEFS = "\n".join(f"- {r.target_label}: {r.definition}" for r in LABELS.itertuples())
TAG_SYS = f"""You label one chat reply. Intent labels:
{LABEL_DEFS}
intended_meaning: what the reply communicates, as a neutral third-person paraphrase of meaning only, e.g. "declines; has a conflict that day; offers a later time".
Never reuse the reply's wording, slang, casing, emoji or hedges. length_bucket: short <=5 words, medium 6-15, long >15."""

def classify_example(context, incoming, reply):
    return llm([{"role": "system", "content": TAG_SYS},
                {"role": "user", "content": f"Context: {context}\nMessage: {incoming}\nReply: {reply}"}],
               schema=TAG_SCHEMA, name="tags")

def ngrams(t, n=4):
    w = re.findall(r"[a-z0-9']+", str(t).lower()); return {tuple(w[i:i+n]) for i in range(len(w)-n+1)}

tagged = {}
for u in USER_IDS:
    rows = pd.concat([db_rows(u), test_rows(u)]).drop_duplicates("row_id")
    for r in rows.itertuples():
        tagged[r.row_id] = classify_example(r.context, r.incoming, r.response)

for u in USER_IDS:
    f = Path(f"frozen/gists_{u}_seed{SEED}.csv")
    if not f.exists():
        t = test_rows(u)
        g = pd.DataFrame({"row_id": t.row_id, "scene_id": t.scene_id, "target_label": t.target_label,
                          "real_reply": t.response, "gist": [tagged[i]["intended_meaning"] for i in t.row_id]})
        g["overlap_4gram"] = [len(ngrams(a) & ngrams(b)) for a, b in zip(g.real_reply, g.gist)]
        g.to_csv(f, index=False)
    g = pd.read_csv(f)
    print(u, "gists:", len(g), "with 4-gram overlap (fix these by hand):", int((g.overlap_4gram > 0).sum()))
GISTS = {u: pd.read_csv(f"frozen/gists_{u}_seed{SEED}.csv").set_index("row_id").gist.to_dict() for u in USER_IDS}

## 5. Style card
Built by `lexicon_gen_judge.build_style_card` from database rows only (test scenes are already removed).

In [ ]:
from lexicon_gen_judge import (build_style_card, build_gen_messages, generate_reply, judge_style_pair, judge_meaning,
                               bootstrap_ci, win_rate, style_win_counts, gist_ok, copy_hits, banned_hits, Ref, BANNED_PHRASES)
from lexicon_retrieval import ExampleStore, embed, strip_sender

def style_stats(replies):
    w = [len(x.split()) for x in replies]
    return {"median_words": float(np.median(w)), "p10": int(np.percentile(w, 10)), "p90": int(np.percentile(w, 90)),
            "final_punct": round(100*np.mean([x.rstrip()[-1] in ".!?" for x in replies])),
            "upper_start": round(100*np.mean([x[0].isupper() for x in replies if x[0].isalpha()]))}

def make_card(uid):
    d = db_rows(uid); reps = d.response.tolist(); used = " ".join(reps).lower()
    neg = d[d.target_label.isin(NEG)].response.tolist()
    never = [p for p in BANNED_PHRASES if p not in used][:8]
    card = build_style_card(reps, negatives_note=f"median {np.median([len(x.split()) for x in neg]):.0f} words, plain, no long explanations",
                            never_says=never, sarcastic_flags=d.is_sarcastic.tolist())
    return {"card": card, "stats": style_stats(reps)}

CARDS = {u: make_card(u) for u in USER_IDS}
for u in USER_IDS: print(u, "\n" + CARDS[u]["card"])

## 6. RAG database
`lexicon_retrieval.ExampleStore`: one store per person, frozen test scenes excluded in both registers, MiniLM for meaning and StyleDistance for style, all on CPU.

In [ ]:
AUG_ROWS = {u: pd.DataFrame() for u in USER_IDS}      # filled by section 10/11 (gpt_confirmed, gpt_edited, agent rows)

def build_store(uid):
    extra = AUG_ROWS[uid] if len(AUG_ROWS[uid]) else None
    return ExampleStore.from_survey(RESPONSE_FILES[uid], BANK_CSV, exclude_scene_ids=SPLITS[uid]["test_scene_ids"],
                                    registers=("chat", "survey") if INCLUDE_SURVEY_REGISTER else ("chat",), extra_rows=extra)

STORES = {u: build_store(u) for u in USER_IDS}
for u in USER_IDS:
    df = STORES[u].df
    assert not df.scene_id.isin(SPLITS[u]["test_scene_ids"]).any(), "test scene leaked into the store"
    print(u, len(df), "records", df.origin.value_counts().to_dict())

## 7. Generation
`lexicon_gen_judge.generate_reply` for every condition. Only the style card and the retrieved references change between conditions. Copy or stock-phrase hits trigger one regeneration, and the hits are logged.

In [ ]:
CONDITIONS = {"A": ("A_plain", False), "B": ("B_semantic", True), "C": ("C_context", True),
              "D": ("D_verified", True), "E": ("E_styledistance", True)}

def get_refs(uid, cond, incoming, context, meaning, label, role):
    name, _ = CONDITIONS[cond]
    rows = STORES[uid].retrieve(name, uid, incoming, k=K, context=context, intended_meaning=meaning,
                                label=label, sender_role=role, return_rows=True)
    if isinstance(rows, list) or len(rows) == 0: return [], []
    refs = [Ref(incoming=r.incoming, reply=r.response, label=str(r.target_label), sender_role=str(r.sender_role), stakes=str(r.stakes))
            for r in rows.itertuples()]
    return refs, rows.scene_id.astype(str).tolist()

def gen_one(uid, row, cond):
    meaning = GISTS[uid][row.row_id]
    incoming = strip_sender(row.incoming)
    refs, ref_ids = get_refs(uid, cond, incoming, row.context, meaning, row.target_label, row.sender_role)
    g = generate_reply(client, uid, meaning, incoming, sender="TM", context=row.context,
                       style_card=CARDS[uid]["card"] if CONDITIONS[cond][1] else None, refs=refs or None,
                       model=MODEL, max_tokens=GEN_MAX_TOKENS, seed=SEED)
    return {"user_id": uid, "row_id": row.row_id, "scene_id": row.scene_id, "label": row.target_label, "cond": cond, "reply": g.reply,
            "regenerated": g.attempts > 1, "copied": bool(g.copy_hits), "banned": bool(g.banned_hits), "ref_scenes": ref_ids}

def has_aug(uid): return STORES[uid].df.origin.isin(["gpt_confirmed", "gpt_edited"]).any()

gens = pd.DataFrame([gen_one(u, r, c) for u in USER_IDS for r in test_rows(u).itertuples()
                     for c in CONDITIONS if c != "D" or has_aug(u)])
for g in gens.itertuples():   # leak check: no retrieved example may be a test scene
    assert not set(g.ref_scenes) & set(SPLITS[g.user_id]["test_scene_ids"]), "LEAK"
gens.to_csv(f"results/generations_seed{SEED}.csv", index=False)
print(gens.groupby("cond").agg(n=("reply", "size"), regen=("regenerated", "mean"), copied=("copied", "mean"),
                               words=("reply", lambda s: s.str.split().str.len().mean())).round(2))
print(f"spend so far ${SPEND['usd']:.4f}")

## 8. Judge calibration
On 20 test items, the real reply vs plain GPT (A). The real reply must not also be the judge's reference, otherwise the test is trivial, so all references here are other database replies. Target: real wins at least 85%.

In [ ]:
def typical(uid, n, exclude=()):
    d = STORES[uid].df[STORES[uid].df.origin == "survey"]
    return d[~d.response.isin(exclude)].sample(n, random_state=SEED).response.tolist()

G = gens.set_index(["row_id", "cond"]).reply.to_dict()
calib = []
for u in USER_IDS:
    for r in test_rows(u).head(20).itertuples():
        ref, *rest = typical(u, 3, exclude=[r.response])
        calib.append(judge_style_pair(client, ref, rest, r.response, G[(r.row_id, "A")], model=MODEL))
rate = np.mean([c == "x" for c in calib])
print(f"Calibration: real beats plain GPT {rate:.0%} (ties {np.mean([c == 'tie' for c in calib]):.0%}); target >= 85% -> {'PASS' if rate >= .85 else 'FIX THE RUBRIC'}")

## 9. Evaluation
Pairwise style score = (wins + 0.5 ties) / n, both orders, with a 95% bootstrap CI (2,000 resamples). A style win with meaning below 4 counts as a tie. StyleDistance similarity to the real reply is backup only (circular for E).

In [ ]:
conds = sorted(gens.cond.unique())
PAIRS = [(c, "A") for c in conds if c != "A"] + [p for p in [("C", "B"), ("E", "C"), ("D", "C")] if p[0] in conds]
rows, mean_rows = [], []
for u in USER_IDS:
    M = {}
    for r in test_rows(u).itertuples():
        g = GISTS[u][r.row_id]; real_vec = embed([r.response], "style")[0]
        for c in conds + ["REAL"]:
            reply = r.response if c == "REAL" else G[(r.row_id, c)]
            M[(r.row_id, c)] = m = judge_meaning(client, g, strip_sender(r.incoming), reply, model=MODEL)
            mean_rows.append({"user_id": u, "row_id": r.row_id, "label": r.target_label, "cond": c, **m,
                              "styledist_cos": float(embed([reply], "style")[0] @ real_vec)})
        ty = typical(u, 2, exclude=[r.response])
        for x, y in PAIRS:
            raw = judge_style_pair(client, r.response, ty, G[(r.row_id, x)], G[(r.row_id, y)], model=MODEL)
            rows.append({"user_id": u, "row_id": r.row_id, "label": r.target_label, "pair": f"{x} vs {y}", "raw": raw,
                         "guarded": style_win_counts(raw, M[(r.row_id, x)], M[(r.row_id, y)])})
pw, mn = pd.DataFrame(rows), pd.DataFrame(mean_rows)
pw.to_csv(f"results/pairwise_seed{SEED}.csv", index=False); mn.to_csv(f"results/meaning_seed{SEED}.csv", index=False)

out = []
for (u, p), g in pw.groupby(["user_id", "pair"]):
    o = g.guarded.tolist(); m, lo, hi = bootstrap_ci(o, seed=SEED); neg = g[g.label.isin(NEG)].guarded.tolist()
    out.append({"user": u, "pair": p, "n": len(o), "win": o.count("x"), "tie": o.count("tie"), "loss": o.count("y"),
                "score": round(m, 2), "ci95": f"[{lo:.2f}, {hi:.2f}]", "score_negative_labels": round(win_rate(neg), 2)})
STYLE_TABLE = pd.DataFrame(out); STYLE_TABLE.to_csv(f"results/style_table_seed{SEED}.csv", index=False)
print("STYLE (guarded by meaning >= 4)"); display(STYLE_TABLE)
mt = mn.groupby(["user_id", "cond"]).agg(meaning=("score", "mean"), act_match=("act_matches", "mean"),
                                        added_facts=("added_facts", "mean"), styledist_cos=("styledist_cos", "mean")).round(2)
mt = mt.join(gens.groupby(["user_id", "cond"]).agg(copy_rate=("copied", "mean"), regen_rate=("regenerated", "mean"),
                                                   words=("reply", lambda s: s.str.split().str.len().mean())).round(2))
mt.to_csv(f"results/guardrails_seed{SEED}.csv"); print("MEANING / GUARDRAILS"); display(mt)
print(f"total spend ${SPEND['usd']:.4f} over {SPEND['calls']} paid calls ({SPEND['cached']} cached)")

In [ ]:
# TF-IDF person classifier: needs at least 2 people. Runs automatically once P2 is in USER_IDS.
if len(USER_IDS) >= 2:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    tr = pd.concat([STORES[u].df[STORES[u].df.origin == "survey"] for u in USER_IDS])
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), lowercase=False)
    clf = LogisticRegression(max_iter=2000).fit(vec.fit_transform(tr.response), tr.user_id)
    gens["p_user"] = [clf.predict_proba(vec.transform([t]))[0][list(clf.classes_).index(u)] for t, u in zip(gens.reply, gens.user_id)]
    display(gens.groupby(["user_id", "cond"]).p_user.agg(["mean", lambda s: (s > 0.5).mean()]).round(2))
else:
    print("Person classifier skipped: only one person loaded.")

## 10. The agent
Tools: `classify_example`, `retrieve`, `generate`, `copy_check`, `style_score`, `judge`, `add_feedback`. Loop: understand, retrieve, draft, check (up to 3 revisions), show, then learn from the accept or edit.

In [ ]:
INTENT_SCHEMA = obj({"intent": {"type": "string", "enum": LABEL_NAMES},
                     "sender_role": {"type": "string", "enum": ["friend", "teammate", "authority"]}})

def style_score(uid, text):
    """Checks a draft against the person's style numbers and returns a list of problems."""
    s = CARDS[uid]["stats"]; w = len(text.split()); probs = []
    if w > max(s["p90"] + 2, 2*s["median_words"]): probs.append(f"too long ({w} words; this person usually writes {s['p10']}-{s['p90']})")
    if s["final_punct"] < 20 and text.rstrip()[-1:] in ".!": probs.append("this person rarely ends with punctuation")
    if s["upper_start"] > 80 and text[:1].islower(): probs.append("this person capitalises the first letter")
    return probs + [f"uses '{p}', a stock phrase" for p in banned_hits(text)]

def add_rows(uid, recs):
    AUG_ROWS[uid] = pd.concat([AUG_ROWS[uid], pd.DataFrame(recs)], ignore_index=True)
    STORES[uid] = build_store(uid)

class LexiconAgent:
    def __init__(self, uid): self.uid = uid
    def understand(self, context, incoming, meaning):
        return llm([{"role": "system", "content": "Pick the intent label the REPLY should have and the sender's role.\n" + LABEL_DEFS},
                    {"role": "user", "content": f"Context: {context}\nMessage: {incoming}\nWhat the reply should say: {meaning}"}],
                   schema=INTENT_SCHEMA, name="intent")
    def reply(self, incoming, meaning, context="Chat", max_revisions=3):
        u = self.understand(context, incoming, meaning)
        cond = "D" if has_aug(self.uid) else "C"
        refs, _ = get_refs(self.uid, cond, strip_sender(incoming), context, meaning, u["intent"], u["sender_role"])
        msgs = build_gen_messages(self.uid, meaning, strip_sender(incoming), "TM", context, CARDS[self.uid]["card"], refs)
        trace = []
        for attempt in range(max_revisions + 1):
            text = llm(msgs, temperature=0.7, max_tokens=GEN_MAX_TOKENS).strip('"')
            probs = [f"copies '{h}'" for h in copy_hits(text, [r.reply for r in refs] + [meaning])] + style_score(self.uid, text)
            trace.append({"draft": text, "problems": probs})
            if not probs: break
            msgs = msgs + [{"role": "assistant", "content": text}, {"role": "user", "content": "Rewrite it. Fix: " + "; ".join(probs)}]
        self.last = {"incoming": strip_sender(incoming), "context": context, **u, "reply": text}
        return text, trace
    def add_feedback(self, edited_text=None):
        """Learn: the accepted (or edited) reply becomes a new database record."""
        L = self.last; final = edited_text or L["reply"]
        add_rows(self.uid, [{"user_id": self.uid, "scene_id": f"agent_{len(AUG_ROWS[self.uid])}", "register": "chat",
                             "incoming": L["incoming"], "response": final, "target_label": L["intent"], "sender_role": L["sender_role"],
                             "stakes": "social", "context": L["context"], "origin": "chatbot_rewrite"}])
        return final

agent = LexiconAgent("P1")
text, trace = agent.reply("TM: can you finish the whole report by tonight?", "says no; can only do their own section")
print(text); print(trace)
if "P2" in USER_IDS:
    print(LexiconAgent("P2").reply("TM: can you finish the whole report by tonight?", "says no; can only do their own section")[0])
# agent.add_feedback()                                  # accept as is
# agent.add_feedback(edited_text="Nah only my part")    # or correct it; either way it joins the database

## 11. Progressive augmentation (free ChatGPT, no API)
1. `export_pack` writes a prompt to paste into ChatGPT. Test scenes are never included.
2. Save ChatGPT's JSON answer as `augment/P1_round1_candidates.json`.
3. `make_review_sheet` writes a CSV. Fill `decision` with confirm, edit or reject, and `edited_reply` for edits.
4. `import_review` adds them as `gpt_confirmed` or `gpt_edited` (rejected rows are kept only as judge negatives). Re-run sections 7 onward to get condition D.

In [ ]:
def export_pack(uid, round_no=1, n_new=20):
    ex = STORES[uid].df[STORES[uid].df.origin == "survey"]
    ex = ex.sample(min(25, len(ex)), random_state=round_no)
    lines = "\n".join(f'- [{r.target_label} | {r.sender_role}] "{r.incoming}" -> "{r.response}"' for r in ex.itertuples())
    pack = f"""Here is how I text. Each line is a message I got and my exact reply:
{lines}

Invent {n_new} NEW short chat messages someone might send me (friends, teammates, a TA or manager), spread across these reply types: {", ".join(LABEL_NAMES)}.
For each, write the reply I would send, copying my length, casing, punctuation and directness exactly. Do not reuse my replies.
Answer only with JSON: [{{"label": "...", "sender_role": "friend|teammate|authority", "context": "...", "incoming_message": "...", "reply": "..."}}]"""
    f = Path(f"augment/{uid}_round{round_no}_pack.txt"); f.write_text(pack); print("wrote", f); return pack

def make_review_sheet(uid, round_no=1):
    c = pd.DataFrame(json.loads(Path(f"augment/{uid}_round{round_no}_candidates.json").read_text()))
    c["decision"], c["edited_reply"] = "", ""
    f = f"augment/{uid}_round{round_no}_review.csv"; c.to_csv(f, index=False); print("fill in", f)

def import_review(uid, round_no=1):
    c = pd.read_csv(f"augment/{uid}_round{round_no}_review.csv").fillna("")
    keep, rejected = [], []
    for i, r in enumerate(c.itertuples()):
        rec = {"user_id": uid, "scene_id": f"aug{round_no}_{i}", "register": "chat", "incoming": strip_sender(r.incoming_message),
               "response": r.edited_reply if r.decision == "edit" else r.reply, "target_label": r.label, "sender_role": r.sender_role,
               "stakes": "social", "context": r.context, "origin": {"confirm": "gpt_confirmed", "edit": "gpt_edited"}.get(r.decision, "gpt_rejected")}
        (keep if r.decision in ("confirm", "edit") else rejected if r.decision == "reject" else []).append(rec)
    add_rows(uid, keep)
    pd.DataFrame(rejected).to_csv(f"augment/{uid}_round{round_no}_rejected.csv", index=False)
    print(f"added {len(keep)} verified rows, kept {len(rejected)} rejected rows as judge negatives")

_ = export_pack("P1", 1)